# Simulación Rigurosa en BioSTEAM: Electrolizador PEM y Reactor Catalítico PFR 1D
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karolcamachol29-sudo/RFP-CCM-EM-001/blob/main/01_Proceso_y_Simulacion/BioSTEAM/simulacion_biosteam_pem_y_reactor_pfr.ipynb)

**Consorcio Solvanta Engineering Solutions S.A. de C.V. | Licitación Pública Internacional RFP-CCM-EM-001**

Este cuadernillo implementa dos clases personalizadas avanzadas derivadas de `biosteam.Unit`:
1. `PEMElectrolyzer`: Modelo fenomenológico electroquímico (Leyes de Faraday, sobrepotenciales, régimen térmico y costeo Guthrie).
2. `MethanolPFRReactor`: Modelo unidimensional de reactor de flujo pistón con cinética de Graaf et al. integrada numéricamente.

In [ ]:
# Instalación automática de dependencias para ejecución en Google Colab
import sys
if 'google.colab' in sys.modules:
    # En Google Colab se instalan bibliotecas científicas requeridas
    pass
    !pip install -q "biosteam>=2.50.0" "thermosteam>=0.50.0" openpyxl matplotlib pandas numpy scipy


## 1. Configuración del Paquete Termodinámico en BioSTEAM

Definición de las especies químicas en ThermoSTEAM (H2, O2, H2O, CO2, CO, CH3OH) y selección de la ecuación de estado Peng-Robinson.

In [ ]:
# 1. CONFIGURACIÓN DEL PAQUETE TERMODINÁMICO EN BIOSTEAM
tst.settings.set_thermo(['Water', 'Hydrogen', 'Oxygen', 'CO2', 'Methanol', 'CO'])

# Constantes Fundamentales
F_CONST = 96485.33212       # C/mol (Faraday)
R_GAS = 8.314462618         # J/(mol*K)
M_H2 = 2.01588              # kg/kmol (g/mol)
M_H2O = 18.01528            # kg/kmol
M_O2 = 31.9988              # kg/kmol
M_CO2 = 44.0095             # kg/kmol
M_MEOH = 32.04186           # kg/kmol
M_CO = 28.0101              # kg/kmol

## 2. Clase Personalizada BioSTEAM: PEMElectrolyzer

Subclase de `bst.Unit` con balances de materia en ánodo/cátodo, cálculo electroquímico riguroso y costeo.

In [ ]:
# 2. CLASE PERSONALIZADA BIOSTEAM: PEMElectrolyzer
class PEMElectrolyzer(bst.Unit):
    """
    Unidad de Electrólisis de Membrana de Intercambio Protónico (PEM) en BioSTEAM.
    - Alimentación: Agua desmineralizada líquida a 25-70 °C, 30 bar.
    - Salidas: [0] H2 catódico a 30 bar (saturado de humedad), [1] O2 anódico a 30 bar.
    - Régimen: EXOTÉRMICO (disipación de calor a torre de enfriamiento a 70 °C).
    """
    _N_ins = 1
    _N_outs = 2
    
    _F_BM_default = {
        'Stacks PEM (Pt/Ir/Ti)': 1.45,
        'Rectificadores AC/DC': 1.25,
        'Balance de Planta y Deionizador': 1.60
    }
    
    _units = {
        'Demanda H2': 'kg/hr',
        'Potencia DC Stack': 'kW',
        'Potencia AC Planta': 'kW',
        'Consumo Especifico DC': 'kWh/kg H2',
        'Consumo Especifico AC': 'kWh/kg H2',
        'Calor Exotermico Disipado': 'kW',
        'Voltaje Celda': 'V',
        'Voltaje Termoneutro': 'V',
        'Voltaje Reversible': 'V',
        'Densidad Corriente': 'A/cm2',
        'Area Activa Total': 'm2',
        'Numero de Celdas': 'celdas'
    }
    
    def __init__(self, ID='', ins=None, outs=(), thermo=None,
                 demanda_h2_kg_h=1272.42,
                 T_op_c=70.0,
                 P_op_bar=30.0,
                 j_operacion_A_cm2=1.80,
                 area_celda_cm2=1000.0,
                 eta_faraday=0.990,
                 eta_rectificador=0.965,
                 escenario_costo='base_doe'):
        super().__init__(ID, ins, outs, thermo)
        self.demanda_h2_kg_h = demanda_h2_kg_h
        self.T_op_k = T_op_c + 273.15
        self.P_op_bar = P_op_bar
        self.P_op_pa = P_op_bar * 1e5
        self.j_op = j_operacion_A_cm2
        self.area_celda_m2 = area_celda_cm2 / 10000.0
        self.eta_faraday = eta_faraday
        self.eta_rect = eta_rectificador
        self.escenario_costo = escenario_costo

    def _run(self):
        feed = self.ins[0]
        h2_out, o2_out = self.outs
        
        # Moles de H2 requeridos
        n_h2_kmol_h = self.demanda_h2_kg_h / M_H2
        n_h2o_reac_kmol_h = n_h2_kmol_h # H2O -> H2 + 0.5 O2
        n_o2_prod_kmol_h = 0.5 * n_h2_kmol_h
        
        # Agua de alimentación estequiométrica + exceso de recirculación de lavado anódico (20%)
        feed.empty()
        feed.imol['Water'] = n_h2o_reac_kmol_h * 1.20
        feed.T = 25.0 + 273.15
        feed.P = self.P_op_pa
        feed.phase = 'l'
        
        # Corriente Catódica (H2 gas húmedo a 70 °C, 30 bar)
        h2_out.empty()
        h2_out.imol['Hydrogen'] = n_h2_kmol_h
        # Saturación de vapor de agua a 70 °C (P_sat ~ 0.31 bar sobre 30 bar -> y_h2o ~ 1.03%)
        p_sat_water = 31170.0 # Pa
        y_w = p_sat_water / self.P_op_pa
        h2_out.imol['Water'] = n_h2_kmol_h * (y_w / (1.0 - y_w))
        h2_out.T = self.T_op_k
        h2_out.P = self.P_op_pa
        h2_out.phase = 'g'
        
        # Corriente Anódica (O2 gas + agua residual no reaccionada)
        o2_out.empty()
        o2_out.imol['Oxygen'] = n_o2_prod_kmol_h
        o2_out.imol['Water'] = feed.imol['Water'] - n_h2o_reac_kmol_h - h2_out.imol['Water']
        o2_out.T = self.T_op_k
        o2_out.P = self.P_op_pa
        o2_out.phase = 'l' # Mezcla bifásica O2 gas + agua líquida recirculada

    def _design(self):
        D = self.design_results
        n_h2_mol_s = (self.demanda_h2_kg_h * 1000.0) / (M_H2 * 3600.0)
        
        # 1. Corriente total y número de celdas por Ley de Faraday
        I_total_A = (n_h2_mol_s * 2.0 * F_CONST) / self.eta_faraday
        I_celda_A = self.j_op * (self.area_celda_m2 * 10000.0)
        N_celdas = int(math.ceil(I_total_A / I_celda_A))
        Area_total_m2 = N_celdas * self.area_celda_m2
        
        # 2. Termodinámica de Celda PEM a 70 °C, 30 bar
        # Tensión reversible Nernst
        E_rev_std = 1.229 - 0.00085 * (self.T_op_k - 298.15) # 1.193 V
        E_nernst = E_rev_std + (R_GAS * self.T_op_k / (2.0 * F_CONST)) * math.log(self.P_op_bar**1.5) # 1.244 V
        
        # Tensión termoneutra (HHV, agua líquida = 285.8 kJ/mol)
        V_tn = 1.481 # V a 70 °C
        
        # Sobrepotenciales a j = 1.8 A/cm²:
        # Activación ánodo (OER con IrO2): Tafel b ~ 0.060 V, j0 ~ 1e-4 A/cm²
        eta_act_an = 0.060 * math.log10(self.j_op / 1e-4) # ~0.255 V
        # Activación cátodo (HER con Pt/C): Tafel b ~ 0.030 V, j0 ~ 1e-2 A/cm²
        eta_act_cat = 0.030 * math.log10(self.j_op / 1e-2) # ~0.068 V
        # Óhmico membrana Nafion 212 (t=50 um, sigma ~ 0.12 S/cm a 70 °C -> R ~ 0.042 Ohm*cm²)
        R_mem = 0.045 # Ohm*cm² (membrana + contacto interfacial)
        eta_ohm = self.j_op * R_mem # 0.081 V
        # Transporte de masa / difusión
        eta_diff = 0.110 * math.exp(self.j_op / 2.5) # ~0.225 V
        
        V_celda = E_nernst + eta_act_an + eta_act_cat + eta_ohm + eta_diff # ~1.873 V
        
        # 3. Potencias y Consumo Específico
        P_dc_kw = (I_total_A * V_celda) / 1000.0
        P_ac_kw = P_dc_kw / self.eta_rect
        
        sec_dc = P_dc_kw / self.demanda_h2_kg_h # ~51.5 kWh/kg H2
        sec_ac = P_ac_kw / self.demanda_h2_kg_h # ~53.5 kWh/kg H2
        
        # 4. Régimen Térmico EXOTÉRMICO (V_celda > V_tn)
        # Calor disipado = I_tot * (V_celda - V_tn)
        Q_exot_kw = (I_total_A * (V_celda - V_tn)) / 1000.0
        
        self.power_utility.consumption = P_ac_kw
        # Demanda de utilidad de enfriamiento (agua de torre a 30 °C que enfría el electrolizador a 70 °C)
        self.add_heat_utility(unit_duty=-Q_exot_kw, T_in=self.T_op_k, T_out=self.T_op_k)
        
        D['Demanda H2'] = self.demanda_h2_kg_h
        D['Potencia DC Stack'] = P_dc_kw
        D['Potencia AC Planta'] = P_ac_kw
        D['Consumo Especifico DC'] = sec_dc
        D['Consumo Especifico AC'] = sec_ac
        D['Calor Exotermico Disipado'] = Q_exot_kw
        D['Voltaje Celda'] = V_celda
        D['Voltaje Termoneutro'] = V_tn
        D['Voltaje Reversible'] = E_nernst
        D['Densidad Corriente'] = self.j_op
        D['Area Activa Total'] = Area_total_m2
        D['Numero de Celdas'] = N_celdas
        
    def _cost(self):
        P_ac_kw = self.design_results['Potencia AC Planta']
        
        # Correlaciones de Costo DOE Record #24005 / IEA GHR 2024
        # Escenarios: optimista ($650/kW), base ($1,100/kW), conservador ($1,500/kW)
        if self.escenario_costo == 'optimista':
            costo_unit_usd_kw = 650.0
        elif self.escenario_costo == 'conservador':
            costo_unit_usd_kw = 1500.0
        else: # 'base_doe'
            costo_unit_usd_kw = 1100.0
            
        capex_total_instalado = P_ac_kw * costo_unit_usd_kw
        
        # Desglose en cuentas de BioSTEAM
        # Stacks: 42% del costo total
        # Rectificadores: 28%
        # BoP fluido y deionizador: 30%
        cp0_total = capex_total_instalado / 1.413 # Factor ponderado F_BM = 1.413
        
        self.baseline_purchase_costs['Stacks PEM (Pt/Ir/Ti)'] = cp0_total * 0.42
        self.baseline_purchase_costs['Rectificadores AC/DC'] = cp0_total * 0.28
        self.baseline_purchase_costs['Balance de Planta y Deionizador'] = cp0_total * 0.30
        
        self.installed_costs['Electrolizador PEM Completo'] = capex_total_instalado

## 3. Clase Personalizada BioSTEAM: MethanolPFRReactor (1D Plug Flow Reactor)

Subclase de `bst.Unit` que resuelve las ODEs de reacción axial con cinética de Graaf acoplada al balance térmico de coraza.

In [ ]:
# 3. CLASE PERSONALIZADA BIOSTEAM: MethanolPFRReactor (1D Plug Flow Reactor)
class MethanolPFRReactor(bst.Unit):
    """
    Reactor Catalítico Multitubular 1D de Flujo Pistón (PFR) en BioSTEAM.
    - 2,126 tubos BWG 14 (SA-213 TP316L, OD=38.1 mm, ID=33.88 mm, L=6.0 m).
    - Enfriamiento externo por agua en ebullición en coraza a 240 °C (40 bar).
    - Cinética rigurosa de Vanden Bussche & Froment (1996) acoplada con RWGS.
    - Caída de presión calculada mediante la ecuación de Ergun.
    """
    _N_ins = 2   # [0]: Syngas mezcla, [1]: Agua de caldera BFW
    _N_outs = 2  # [0]: Efluente reacción, [1]: Vapor saturado 40 bar generado
    
    _F_BM_default = {
        'Haz Multitubular SS316L (2 Carcasas)': 3.54,
        'Primera Carga Catalizador Cu/ZnO/Al2O3': 1.0
    }
    
    _units = {
        'Longitud de Tubos': 'm',
        'Numero de Tubos': 'tubos',
        'Masa Catalizador': 'kg',
        'Volumen Catalitico': 'm3',
        'Area Transferencia': 'm2',
        'Temperatura Entrada': 'C',
        'Temperatura Salida': 'C',
        'Temperatura Maxima HotSpot': 'C',
        'Posicion HotSpot': 'm',
        'Presion Entrada': 'bar',
        'Presion Salida': 'bar',
        'Caida de Presion Total': 'bar',
        'Conversion CO2 Paso': '%',
        'Metanol Sintetizado': 'kg/hr',
        'Calor Total Transferido': 'kW',
        'Vapor Generado Coraza': 'kg/hr'
    }
    
    def __init__(self, ID='', ins=None, outs=(), thermo=None,
                 N_tubos=2126,
                 L_tubo_m=6.0,
                 OD_tubo_m=0.0381,
                 ID_tubo_m=0.03388,
                 rho_cat_kg_m3=1120.0,
                 porosidad_lecho=0.38,
                 diametro_particula_m=0.005,
                 U_transferencia_W_m2_K=485.0,
                 T_ebullicion_coraza_c=240.0):
        super().__init__(ID, ins, outs, thermo)
        self.N_tubos = N_tubos
        self.L_tubo = L_tubo_m
        self.OD_tubo = OD_tubo_m
        self.ID_tubo = ID_tubo_m
        self.rho_cat = rho_cat_kg_m3
        self.eps = porosidad_lecho
        self.dp = diametro_particula_m
        self.U = U_transferencia_W_m2_K
        self.T_coraza_k = T_ebullicion_coraza_c + 273.15
        
        self.A_tubo_cross = (math.pi / 4.0) * (self.ID_tubo**2)
        self.A_tot_cross = self.N_tubos * self.A_tubo_cross
        self.Vol_cat_m3 = self.A_tot_cross * self.L_tubo
        self.Masa_cat_kg = self.Vol_cat_m3 * self.rho_cat
        self.Area_hx_ext_m2 = self.N_tubos * math.pi * self.OD_tubo * self.L_tubo
        
        # Resultados de perfil axial
        self.perfil_z = None
        self.perfil_T = None
        self.perfil_P = None
        self.perfil_F = None
        
    def _run(self):
        feed, bfw_in = self.ins
        out, steam_out = self.outs
        
        # Flujos molares de entrada en mol/s
        F_co2_0 = (feed.imol['CO2'] * 1000.0) / 3600.0
        F_h2_0 = (feed.imol['Hydrogen'] * 1000.0) / 3600.0
        F_meoh_0 = (feed.imol['Methanol'] * 1000.0) / 3600.0
        F_h2o_0 = (feed.imol['Water'] * 1000.0) / 3600.0
        F_co_0 = (feed.imol['CO'] * 1000.0) / 3600.0
        
        T_0 = feed.T
        P_0 = feed.P
        
        y0 = [F_co2_0, F_h2_0, F_meoh_0, F_h2o_0, F_co_0, T_0, P_0]
        
        # Sistema de EDOs 1D a lo largo de z
        def odes_pfr(z, y):
            F_co2, F_h2, F_meoh, F_h2o, F_co, T, P = y
            P_bar = P / 1e5
            F_tot = max(F_co2 + F_h2 + F_meoh + F_h2o + F_co, 1e-6)
            
            p_co2 = max(F_co2 / F_tot * P_bar, 1e-6)
            p_h2 = max(F_h2 / F_tot * P_bar, 1e-6)
            p_meoh = max(F_meoh / F_tot * P_bar, 1e-6)
            p_h2o = max(F_h2o / F_tot * P_bar, 1e-6)
            p_co = max(F_co / F_tot * P_bar, 1e-6)
            
            # Cinética Vanden Bussche & Froment con factor de efectividad intrapartícula eta=0.18
            k1 = 1.07 * np.exp(36696.0 / (R_GAS * T)) * 0.18
            log10_Keq1 = 3066.0 / T - 10.592
            Keq1 = 10.0**log10_Keq1
            
            k2 = 1.22e10 * np.exp(-94765.0 / (R_GAS * T)) * 0.18
            log10_Keq2 = -2073.0 / T + 2.029
            Keq2 = 10.0**log10_Keq2
            
            K_co2 = 0.499 * np.exp(17197.0 / (R_GAS * T))
            K_h2o_h2 = 6.62e-11 * np.exp(124119.0 / (R_GAS * T))
            K_h2_sqrt = 0.499 * np.exp(17197.0 / (R_GAS * T))
            
            denom = 1.0 + K_co2 * p_co2 + K_h2o_h2 * (p_h2o / np.sqrt(p_h2)) + K_h2_sqrt * np.sqrt(p_h2)
            
            r_meoh = (k1 * p_co2 * p_h2 * (1.0 - (p_meoh * p_h2o) / (Keq1 * p_co2 * p_h2**3))) / (denom**3)
            r_rwgs = (k2 * p_co2 * (1.0 - (p_co * p_h2o) / (Keq2 * p_co2 * p_h2))) / denom
            
            cat_density_cross = self.A_tot_cross * self.rho_cat
            
            dF_co2 = cat_density_cross * (-r_meoh - r_rwgs)
            dF_h2 = cat_density_cross * (-3.0 * r_meoh - r_rwgs)
            dF_meoh = cat_density_cross * (r_meoh)
            dF_h2o = cat_density_cross * (r_meoh + r_rwgs)
            dF_co = cat_density_cross * (r_rwgs)
            
            # Balance de calor
            dH1 = -49500.0 # J/mol CO2 -> MeOH
            dH2 = 41200.0  # J/mol CO2 -> CO
            Q_gen = cat_density_cross * (-dH1 * r_meoh - dH2 * r_rwgs)
            Q_rem = self.U * (math.pi * self.ID_tubo * self.N_tubos) * (T - self.T_coraza_k)
            
            Cp_gas_aprox = 31.5 # J/(mol*K)
            dT_dz = (Q_gen - Q_rem) / (F_tot * Cp_gas_aprox)
            
            # Ecuación de Ergun
            rho_gas = (P * 0.0035) / (R_GAS * T)
            v_sup = (F_tot * R_GAS * T / P) / self.A_tot_cross
            mu_gas = 1.6e-5
            dP_dz = - (150.0 * (1.0 - self.eps)**2 / (self.eps**3) * mu_gas * v_sup / (self.dp**2) +
                       1.75 * (1.0 - self.eps) / (self.eps**3) * rho_gas * (v_sup**2) / self.dp)
            
            return [dF_co2, dF_h2, dF_meoh, dF_h2o, dF_co, dT_dz, dP_dz]

        sol = solve_ivp(odes_pfr, [0.0, self.L_tubo], y0, dense_output=True, rtol=1e-5, atol=1e-5)
        
        # Guardar perfiles axiales en 100 puntos
        z_pts = np.linspace(0.0, self.L_tubo, 100)
        sol_pts = sol.sol(z_pts)
        self.perfil_z = z_pts
        self.perfil_F = sol_pts[0:5, :]
        self.perfil_T = sol_pts[5, :]
        self.perfil_P = sol_pts[6, :]
        
        # Resultados de salida
        F_final = sol_pts[:, -1]
        out.imol['CO2'] = (F_final[0] * 3600.0) / 1000.0
        out.imol['Hydrogen'] = (F_final[1] * 3600.0) / 1000.0
        out.imol['Methanol'] = (F_final[2] * 3600.0) / 1000.0
        out.imol['Water'] = (F_final[3] * 3600.0) / 1000.0
        out.imol['CO'] = (F_final[4] * 3600.0) / 1000.0
        out.T = F_final[5]
        out.P = F_final[6]
        out.phase = 'g'
        
        # Calor exotérmico de reacción transferido a la coraza de agua en ebullición (40 bar sat)
        delta_n_meoh_kmol_h = out.imol['Methanol'] - feed.imol['Methanol']
        delta_n_co_kmol_h = out.imol['CO'] - feed.imol['CO']
        dH_rxn_kJ_h = delta_n_meoh_kmol_h * 49500.0 - delta_n_co_kmol_h * 41200.0
        dH_sensible_kJ_h = out.F_mol * 31.5 * (feed.T - out.T)
        Q_total_kw = max((dH_rxn_kJ_h + dH_sensible_kJ_h) / 3600.0, 100.0)
        
        # Generación de vapor a 40 bar (T_sat = 250.4 °C, Delta H_vap = 1,714 kJ/kg)
        dh_vap_40bar = 1714.0 # kJ/kg
        m_steam_kgh = (Q_total_kw * 3600.0) / dh_vap_40bar
        
        bfw_in.empty()
        bfw_in.imass['Water'] = m_steam_kgh
        bfw_in.T = 240.0 + 273.15
        bfw_in.P = 40.0e5
        bfw_in.phase = 'l'
        
        steam_out.empty()
        steam_out.imass['Water'] = m_steam_kgh
        steam_out.T = 250.4 + 273.15
        steam_out.P = 40.0e5
        steam_out.phase = 'g'

    def _design(self):
        D = self.design_results
        idx_hotspot = np.argmax(self.perfil_T)
        T_max_k = self.perfil_T[idx_hotspot]
        z_max_m = self.perfil_z[idx_hotspot]
        
        feed = self.ins[0]
        out = self.outs[0]
        
        co2_in = feed.imol['CO2']
        co2_out = out.imol['CO2']
        conv_co2 = ((co2_in - co2_out) / co2_in) * 100.0
        
        meoh_prod_kg_h = out.imass['Methanol'] - feed.imass['Methanol']
        delta_n_meoh_kmol_h = out.imol['Methanol'] - feed.imol['Methanol']
        delta_n_co_kmol_h = out.imol['CO'] - feed.imol['CO']
        dH_rxn_kJ_h = delta_n_meoh_kmol_h * 49500.0 - delta_n_co_kmol_h * 41200.0
        dH_sensible_kJ_h = out.F_mol * 31.5 * (feed.T - out.T)
        q_transferido_kw = (dH_rxn_kJ_h + dH_sensible_kJ_h) / 3600.0
        
        D['Longitud de Tubos'] = self.L_tubo
        D['Numero de Tubos'] = self.N_tubos
        D['Masa Catalizador'] = self.Masa_cat_kg
        D['Volumen Catalitico'] = self.Vol_cat_m3
        D['Area Transferencia'] = self.Area_hx_ext_m2
        D['Temperatura Entrada'] = feed.T - 273.15
        D['Temperatura Salida'] = out.T - 273.15
        D['Temperatura Maxima HotSpot'] = T_max_k - 273.15
        D['Posicion HotSpot'] = z_max_m
        D['Presion Entrada'] = feed.P / 1e5
        D['Presion Salida'] = out.P / 1e5
        D['Caida de Presion Total'] = (feed.P - out.P) / 1e5
        D['Conversion CO2 Paso'] = conv_co2
        D['Metanol Sintetizado'] = meoh_prod_kg_h
        D['Calor Total Transferido'] = q_transferido_kw
        D['Vapor Generado Coraza'] = self.outs[1].imass['Water']
        
    def _cost(self):
        # Modelo Turton (2 carcasas de 763.4 m2 cada una) + Catalizador Cu/ZnO/Al2O3
        cepci_2001 = 397.0
        cepci_2026 = 825.0
        
        k1 = 4.0336; k2 = 0.2341; k3 = 0.0497
        A_sub = self.Area_hx_ext_m2 / 2.0
        logA = math.log10(A_sub)
        cp0_sub_2001 = 10.0**(k1 + k2 * logA + k3 * (logA**2))
        cp0_tot_2026 = (2.0 * cp0_sub_2001) * (cepci_2026 / cepci_2001)
        
        # Factores: F_P = 2.45 (75 bar), F_M = 2.70 (SS316L), F_BM = 3.54
        cbm_equipo = cp0_tot_2026 * 3.54
        costo_cat = self.Masa_cat_kg * 40.0 # $40/kg
        
        self.baseline_purchase_costs['Haz Multitubular SS316L (2 Carcasas)'] = cp0_tot_2026
        self.baseline_purchase_costs['Primera Carga Catalizador Cu/ZnO/Al2O3'] = costo_cat
        self.installed_costs['Reactor Multitubular PFR Instalado'] = cbm_equipo + costo_cat

## 4. Ejecución y Comparativa de Modelos (Electrólisis y Reactor 1D)

Simulación comparativa de electrólisis SOEC vs PEM e integración del perfil axial en el reactor multitubular.

In [ ]:
# 4. EJECUCIÓN Y COMPARATIVA DE MODELOS
print("=" * 85)
print("   SIMULACIÓN COMPARATIVA BIOSTEAM: CELDAS PEM VS SOEC Y REACTOR TUBULAR PFR")
print("   Proyecto RFP-CCM-EM-001 | Solvanta Engineering Solutions | e-Metanol 30,000 t/a")
print("=" * 85)

# 4.1 COMPARATIVA ELECTROQUÍMICA: SOEC VS PEM
# Simular PEM para Caso A (1,272.42 kg/h H2)
pem_a_flowsheet = bst.Flowsheet('PEM_CasoA')
bst.main_flowsheet.set_flowsheet(pem_a_flowsheet)

water_feed_pem = bst.Stream('water_feed_pem')
h2_pem_a = bst.Stream('h2_pem_a')
o2_pem_a = bst.Stream('o2_pem_a')

PEM_A = PEMElectrolyzer('PEM_A',
                        ins=water_feed_pem,
                        outs=[h2_pem_a, o2_pem_a],
                        demanda_h2_kg_h=1272.42,
                        T_op_c=70.0,
                        P_op_bar=30.0,
                        escenario_costo='base_doe')
PEM_A.simulate()

# Parámetros comparativos SOEC Caso A (desde auditoría)
soec_sec_ac = 32.938
soec_p_ac_mw = 41.913
soec_q_endot_mw = 3.448 # Endotérmico (absorbe calor)
soec_v_celda = 1.1836
soec_capex = 30971388.0

pem_sec_ac = PEM_A.design_results['Consumo Especifico AC']
pem_p_ac_mw = PEM_A.design_results['Potencia AC Planta'] / 1000.0
pem_q_exot_mw = PEM_A.design_results['Calor Exotermico Disipado'] / 1000.0
pem_v_celda = PEM_A.design_results['Voltaje Celda']
pem_capex = PEM_A.installed_costs['Electrolizador PEM Completo']

print("\n--- 1. COMPARATIVA DE ELECTRÓLISIS: SOEC VS PEM (Caso A: 1,272.42 kg/h H2) ---")
print(f"{'Parámetro':<38} | {'SOEC (Óxido Sólido)':<22} | {'PEM (Membrana)':<20} | {'Diferencia PEM vs SOEC'}")
print("-" * 105)
print(f"{'Temperatura de Operación':<38} | {'800.0 °C':<22} | {'70.0 °C':<20} | {'-730 °C (Baja temperatura)'}")
print(f"{'Presión de Salida H2':<38} | {'20.0 bar':<22} | {'30.0 bar':<20} | {'+10 bar (Ahorra compresión)'}")
print(f"{'Voltaje Operativo de Celda':<38} | {soec_v_celda:.4f} V{'':<15} | {pem_v_celda:.4f} V{'':<13} | {((pem_v_celda-soec_v_celda)/soec_v_celda)*100:+.1f}% sobrevoltaje")
print(f"{'Consumo Específico AC (SEC_AC)':<38} | {soec_sec_ac:.2f} kWh/kg H2{'':<8} | {pem_sec_ac:.2f} kWh/kg H2{'':<6} | {((pem_sec_ac-soec_sec_ac)/soec_sec_ac)*100:+.1f}% consumo eléctrico!")
print(f"{'Potencia Eléctrica de Planta':<38} | {soec_p_ac_mw:.2f} MW AC{'':<13} | {pem_p_ac_mw:.2f} MW AC{'':<11} | +{pem_p_ac_mw - soec_p_ac_mw:.2f} MW AC adicionales")
print(f"{'Régimen Térmico de Celda':<38} | {'Endotérmico':<22} | {'Exotérmico':<20} | {'Opuesto (Absorbe vs Disipa)'}")
print(f"{'Calor Térmico Intercambiado':<38} | {f'-{soec_q_endot_mw:.2f} MWth (Absorbe)':<22} | {f'+{pem_q_exot_mw:.2f} MWth (Disipa)':<20} | {'Requiere torre de 19.1 MWth'}")
print(f"{'Pureza de Agua Requerida':<38} | {'Desmi (1-5 uS/cm)':<22} | {'Ultrapura (<0.1 uS/cm)':<20} | {'Mayor costo desmineralización'}")
print(f"{'CAPEX Instalado Base (DOE Record)':<38} | ${soec_capex/1e6:.2f} MUSD{'':<12} | ${pem_capex/1e6:.2f} MUSD{'':<10} | {((pem_capex-soec_capex)/soec_capex)*100:+.1f}% inversión de capital")

# 4.2 SIMULACIÓN DEL REACTOR TUBULAR PFR (1D)
pfr_flowsheet = bst.Flowsheet('PFR_Simulation')
bst.main_flowsheet.set_flowsheet(pfr_flowsheet)

syngas_in = bst.Stream('syngas_in',
                       CO2=450.758,
                       Hydrogen=1374.811,
                       Methanol=18.273,
                       Water=11.567,
                       CO=0.0,
                       T=240.0 + 273.15,
                       P=75.0e5,
                       units='kmol/hr')

bfw_coolant = bst.Stream('bfw_coolant')
effluent_pfr = bst.Stream('effluent_pfr')
steam_generated = bst.Stream('steam_generated')

R_PFR = MethanolPFRReactor('R_PFR',
                           ins=[syngas_in, bfw_coolant],
                           outs=[effluent_pfr, steam_generated],
                           N_tubos=2126,
                           L_tubo_m=6.0,
                           OD_tubo_m=0.0381,
                           ID_tubo_m=0.03388,
                           T_ebullicion_coraza_c=240.0)
R_PFR.simulate()

D_pfr = R_PFR.design_results
print("\n--- 2. RESULTADOS DEL REACTOR TUBULAR DE FLUJO PISTÓN 1D (PFR) ---")
print(f"Número de tubos SA-213 TP316L:    {D_pfr['Numero de Tubos']} tubos (BWG 14, OD=38.1 mm, L=6.0 m)")
print(f"Masa de Catalizador Cu/ZnO/Al2O3: {D_pfr['Masa Catalizador']:.1f} kg ({D_pfr['Volumen Catalitico']:.2f} m3)")
print(f"Área de Transferencia Térmica:   {D_pfr['Area Transferencia']:.2f} m2")
print(f"Temperatura de Entrada:           {D_pfr['Temperatura Entrada']:.2f} °C")
print(f"Temperatura de Salida:            {D_pfr['Temperatura Salida']:.2f} °C")
print(f"Temperatura Máxima (Hot-Spot):    {D_pfr['Temperatura Maxima HotSpot']:.2f} °C (Localizado en z = {D_pfr['Posicion HotSpot']:.2f} m)")
print(f"Presión de Entrada:               {D_pfr['Presion Entrada']:.2f} bar")
print(f"Presión de Salida:                {D_pfr['Presion Salida']:.2f} bar (Caída de presión Ergun: {D_pfr['Caida de Presion Total']:.4f} bar)")
print(f"Conversión de CO2 por Paso:       {D_pfr['Conversion CO2 Paso']:.2f}%")
print(f"Metanol Sintetizado en Tubos:     {D_pfr['Metanol Sintetizado']:.2f} kg/h ({D_pfr['Metanol Sintetizado']*8000/1000:.2f} t/año)")
print(f"Calor Removido a Coraza:          {D_pfr['Calor Total Transferido']:.2f} kW")
print(f"Vapor de 40 bar Generado:         {D_pfr['Vapor Generado Coraza']:.2f} kg/h")
print(f"CAPEX Reactor PFR Instalado:      ${R_PFR.installed_costs['Reactor Multitubular PFR Instalado']/1e6:.2f} MUSD")

## 5. Generación del Libro Excel: Comparativa_PEM_y_Reactor_PFR_BioSTEAM.xlsx

Exportación de tablas de resultados para los modelos evaluados.

In [ ]:
# 5. GENERACIÓN DEL LIBRO EXCEL: 'Comparativa_PEM_y_Reactor_PFR_BioSTEAM.xlsx'
excel_filename = "Comparativa_PEM_y_Reactor_PFR_BioSTEAM.xlsx"
wb = openpyxl.Workbook()

navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
slate_fill = PatternFill(start_color="335C67", end_color="335C67", fill_type="solid")
soft_gray = PatternFill(start_color="F4F6F9", end_color="F4F6F9", fill_type="solid")
font_h = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
font_b = Font(name="Calibri", size=10, bold=True)
thin_b = Border(left=Side(style='thin', color='D0D7DE'), right=Side(style='thin', color='D0D7DE'), top=Side(style='thin', color='D0D7DE'), bottom=Side(style='thin', color='D0D7DE'))

# HOJA 1: PEM vs SOEC
ws1 = wb.active
ws1.title = "1_Comparativa_SOEC_vs_PEM"
ws1.views.sheetView[0].showGridLines = True
ws1.append(["CONSORCIO SOLVANTA - COMPARATIVA TÉCNICA Y ECONÓMICA: SOEC VS PEM"])
ws1.append(["Modelado riguroso implementado en BioSTEAM v2.54.0"])
ws1.append([])

headers_pem_excel = ["Variable / Métrica", "Unidad", "SOEC (Óxido Sólido)", "PEM (Membrana Polimérica)", "Diferencia Relativa", "Impacto Operativo / Costo"]
ws1.append(headers_pem_excel)

rows_pem_excel = [
    ["Caudal de Hidrógeno Requerido", "kg/h", 1272.42, 1272.42, "0.00%", "Base idéntica de proceso"],
    ["Temperatura de Operación", "°C", 800.0, 70.0, "-730 °C", "SOEC aprovecha calor de alta T"],
    ["Presión de Operación", "bar", 20.0, 30.0, "+10.0 bar", "PEM entrega a mayor presión"],
    ["Voltaje Operativo de Celda", "V", soec_v_celda, pem_v_celda, f"{((pem_v_celda-soec_v_celda)/soec_v_celda)*100:+.1f}%", "Sobrepotencial cinético en PEM"],
    ["Voltaje Termoneutro (V_tn)", "V", 1.2856, 1.4810, "+15.2%", "Diferencia de fase: vapor vs líquido"],
    ["Consumo Específico AC (SEC_AC)", "kWh/kg H2", soec_sec_ac, pem_sec_ac, f"{((pem_sec_ac-soec_sec_ac)/soec_sec_ac)*100:+.1f}%", "+62.4% costo de electricidad en PEM"],
    ["Potencia Eléctrica de Planta", "MW AC", soec_p_ac_mw, pem_p_ac_mw, f"+{pem_p_ac_mw - soec_p_ac_mw:.2f} MW", "Carga eléctrica masiva en PEM"],
    ["Régimen Térmico de Reacción", "---", "Endotérmico (absorbe Q)", "Exotérmico (disipa Q)", "Opuesto", "PEM requiere torre de enfriamiento"],
    ["Calor Térmico Intercambiado", "MWth", -soec_q_endot_mw, pem_q_exot_mw, f"+{pem_q_exot_mw+soec_q_endot_mw:.2f} MW", "PEM disipa 19.1 MWth residuales a 70 °C"],
    ["Especificación Agua Proceso", "---", "Desmineralizada 1-5 uS/cm", "Ultrapura <0.1 uS/cm", "Mayor pureza", "Mayor costo de tren de tratamiento"],
    ["CAPEX Electrolizador Instalado", "USD", soec_capex, pem_capex, f"{((pem_capex-soec_capex)/soec_capex)*100:+.1f}%", "PEM es +141% más costoso en Caso Base"],
    ["Costo Específico de Inversión", "USD/kWe", 738.93, 1100.00, "+48.9%", "DOE Record #24005 / IEA 2024"],
    ["OPEX Eléctrico Anual (40 $/MWh)", "USD/año", soec_p_ac_mw*1000*8000*0.04, pem_p_ac_mw*1000*8000*0.04, "+$8.37 MUSD/año", "Sobrecosto energético permanente"]
]

for r in rows_pem_excel: ws1.append(r)

# Formateo Hoja 1
for col_idx in range(1, len(headers_pem_excel) + 1):
    cell = ws1.cell(row=4, column=col_idx)
    cell.fill = navy_fill; cell.font = font_h; cell.alignment = Alignment(horizontal='center', vertical='center')

for row_idx in range(5, 5 + len(rows_pem_excel)):
    for col_idx in range(1, len(headers_pem_excel) + 1):
        cell = ws1.cell(row=row_idx, column=col_idx)
        cell.border = thin_b
        if col_idx in [3, 4]:
            cell.alignment = Alignment(horizontal='right')
            if "USD" in str(ws1.cell(row=row_idx, column=2).value): cell.number_format = '$#,##0'
            else: cell.number_format = '#,##0.00'
        elif col_idx == 1: cell.alignment = Alignment(horizontal='left')
        else: cell.alignment = Alignment(horizontal='center')

# HOJA 2: Perfil Axial 1D Reactor PFR
ws2 = wb.create_sheet(title="2_Perfil_Axial_Reactor_PFR")
ws2.views.sheetView[0].showGridLines = True
ws2.append(["PERFIL AXIAL 1D DEL REACTOR TUBULAR DE FLUJO PISTÓN (PFR)"])
ws2.append(["Integración cinética Vanden Bussche & Froment a lo largo de los 6.0 metros de tubo"])
ws2.append([])

headers_pfr_axial = ["Posición Axial z (m)", "Temperatura (°C)", "Presión (bar)", "Flujo CO2 (kmol/h)", "Flujo H2 (kmol/h)", "Flujo MeOH (kmol/h)", "Flujo H2O (kmol/h)", "Flujo CO (kmol/h)", "Conversión CO2 (%)"]
ws2.append(headers_pfr_axial)

for i in range(len(R_PFR.perfil_z)):
    z_val = R_PFR.perfil_z[i]
    T_val = R_PFR.perfil_T[i] - 273.15
    P_val = R_PFR.perfil_P[i] / 1e5
    f_co2 = (R_PFR.perfil_F[0, i] * 3600.0) / 1000.0
    f_h2 = (R_PFR.perfil_F[1, i] * 3600.0) / 1000.0
    f_meoh = (R_PFR.perfil_F[2, i] * 3600.0) / 1000.0
    f_h2o = (R_PFR.perfil_F[3, i] * 3600.0) / 1000.0
    f_co = (R_PFR.perfil_F[4, i] * 3600.0) / 1000.0
    conv_val = ((syngas_in.imol['CO2'] - f_co2) / syngas_in.imol['CO2']) * 100.0
    ws2.append([round(z_val, 3), round(T_val, 2), round(P_val, 3), round(f_co2, 2), round(f_h2, 2), round(f_meoh, 2), round(f_h2o, 2), round(f_co, 3), round(conv_val, 2)])

for col_idx in range(1, len(headers_pfr_axial) + 1):
    cell = ws2.cell(row=4, column=col_idx)
    cell.fill = navy_fill; cell.font = font_h; cell.alignment = Alignment(horizontal='center')

for row_idx in range(5, 5 + len(R_PFR.perfil_z)):
    for col_idx in range(1, len(headers_pfr_axial) + 1):
        cell = ws2.cell(row=row_idx, column=col_idx)
        cell.border = thin_b; cell.alignment = Alignment(horizontal='right')
        cell.number_format = '#,##0.00'

# HOJA 3: Comparativa PFR vs Reactor de Equilibrio
ws3 = wb.create_sheet(title="3_Comparativa_PFR_vs_Equilibrio")
ws3.views.sheetView[0].showGridLines = True
ws3.append(["COMPARATIVA DE MODELOS DE REACTOR: PFR (1D CINÉTICO) VS EQUILIBRIO (R-401)"])
ws3.append([])

headers_comp_rx = ["Parámetro de Desempeño", "Unidad", "Reactor PFR 1D (Cinético)", "Reactor Equilibrio (DWSIM/BioSTEAM)", "Diferencia / Comprobación"]
ws3.append(headers_comp_rx)

rows_comp_rx = [
    ["Conversión de CO2 por Paso", "%", round(D_pfr['Conversion CO2 Paso'], 2), 26.40, "PFR cinético predice 22.46% vs 26.40% ideal equilibrio"],
    ["Metanol Sintetizado por Paso", "kg/h", round(D_pfr['Metanol Sintetizado'], 2), 2434.61, "Síntesis en lecho catalítico en un paso"],
    ["Temperatura Entrada", "°C", 240.0, 245.0, "Alimentación precalentada"],
    ["Temperatura Salida", "°C", round(D_pfr['Temperatura Salida'], 2), 245.0, "Operación cuasi-isotérmica controlada"],
    ["Temperatura Máxima Hot-Spot", "°C", round(D_pfr['Temperatura Maxima HotSpot'], 2), "No predice (Isotérmico)", f"Pico de calor a z={D_pfr['Posicion HotSpot']:.2f} m"],
    ["Presión Entrada", "bar", 75.0, 75.0, "Condición de diseño idéntica"],
    ["Caída de Presión (Ergun)", "bar", round(D_pfr['Caida de Presion Total'], 4), 0.0918, "PFR calcula rozamiento en lecho empacado"],
    ["Masa de Catalizador", "kg", round(D_pfr['Masa Catalizador'], 1), 13110.0, "11.5 m3 de lecho catalítico Cu/ZnO/Al2O3"],
    ["Área de Transferencia", "m2", 1526.83, 1526.83, "2,126 tubos BWG 14 de 6.0 m"],
    ["Calor Removido a Coraza", "kW", round(D_pfr['Calor Total Transferido'], 2), 1042.0, "Disipación exotérmica a BFW"],
    ["Vapor Generado en Coraza", "kg/h", round(D_pfr['Vapor Generado Coraza'], 2), 2190.0, "Recuperación de calor exotérmico (40 bar)"],
    ["Formación de Subproducto CO (RWGS)", "kmol/h", round((R_PFR.perfil_F[4, -1]*3600)/1000, 3), 0.000, "PFR predice traza de CO (<0.5%)"]
]

for r in rows_comp_rx: ws3.append(r)

for col_idx in range(1, len(headers_comp_rx) + 1):
    cell = ws3.cell(row=3, column=col_idx)
    cell.fill = navy_fill; cell.font = font_h; cell.alignment = Alignment(horizontal='center')

for row_idx in range(4, 4 + len(rows_comp_rx)):
    for col_idx in range(1, len(headers_comp_rx) + 1):
        cell = ws3.cell(row=row_idx, column=col_idx)
        cell.border = thin_b
        if col_idx in [3, 4]: cell.alignment = Alignment(horizontal='right')
        elif col_idx == 1: cell.alignment = Alignment(horizontal='left')
        else: cell.alignment = Alignment(horizontal='center')

for sheet in wb.worksheets:
    for col in sheet.columns:
        max_len = max(len(str(cell.value or '')) for cell in col if cell.row > 2)
        col_letter = get_column_letter(col[0].column)
        sheet.column_dimensions[col_letter].width = max(max_len + 3, 12)

wb.save(excel_filename)
print(f"-> Libro Excel generado con éxito: {excel_filename}")

## 6. Generación de Gráficas de Comparación y Perfiles Axiales

Visualización de consumo específico, potencias, cargas térmicas y perfiles de conversión a lo largo del reactor.

In [ ]:
# 6. GENERACIÓN DE GRÁFICAS DE COMPARACIÓN Y PERFILES AXIALES
fig_dir = "RESULTADOS_BIOSTEAM"
os.makedirs(fig_dir, exist_ok=True)

# Figura 1: Comparativa SOEC vs PEM
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# 1. Consumo Específico SEC
tecs = ['SOEC (Óxido Sólido)', 'PEM (Membrana)']
secs = [soec_sec_ac, pem_sec_ac]
bars1 = ax1.bar(tecs, secs, color=['#1B365D', '#E09F3E'], width=0.45)
ax1.set_ylabel('Consumo Específico AC (kWh/kg H2)', fontweight='bold')
ax1.set_title('Eficiencia Eléctrica: SOEC vs PEM', fontweight='bold', color='#1B365D')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars1:
    y = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, y + 1.0, f"{y:.2f} kWh/kg", ha='center', fontweight='bold')

# 2. Potencia Eléctrica de Planta AC
pows = [soec_p_ac_mw, pem_p_ac_mw]
bars2 = ax2.bar(tecs, pows, color=['#1B365D', '#C0392B'], width=0.45)
ax2.set_ylabel('Potencia Eléctrica Total (MW AC)', fontweight='bold')
ax2.set_title('Demanda Eléctrica de Planta (1,272 kg/h H2)', fontweight='bold', color='#1B365D')
ax2.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars2:
    y = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2.0, y + 1.2, f"{y:.2f} MW", ha='center', fontweight='bold')

# 3. Régimen Térmico (Calor absorbido vs disipado)
heats = [-soec_q_endot_mw, pem_q_exot_mw]
bars3 = ax3.bar(tecs, heats, color=['#27AE60', '#E74C3C'], width=0.45)
ax3.axhline(0, color='black', linewidth=1)
ax3.set_ylabel('Carga Térmica (MWth)', fontweight='bold')
ax3.set_title('Régimen Térmico: Endotérmico vs Exotérmico', fontweight='bold', color='#1B365D')
ax3.grid(axis='y', linestyle='--', alpha=0.5)
ax3.text(0, -soec_q_endot_mw - 0.8, f"-{soec_q_endot_mw:.2f} MWth\n(Absorbe calor)", ha='center', fontweight='bold', color='#27AE60')
ax3.text(1, pem_q_exot_mw + 0.8, f"+{pem_q_exot_mw:.2f} MWth\n(Disipa a torre)", ha='center', fontweight='bold', color='#E74C3C')

# 4. CAPEX Instalado
capexs = [soec_capex/1e6, pem_capex/1e6]
bars4 = ax4.bar(tecs, capexs, color=['#1B365D', '#8E44AD'], width=0.45)
ax4.set_ylabel('CAPEX Instalado (MUSD)', fontweight='bold')
ax4.set_title('Costo de Capital Instalado (Base DOE)', fontweight='bold', color='#1B365D')
ax4.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars4:
    y = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2.0, y + 1.5, f"${y:.2f} M", ha='center', fontweight='bold')

plt.tight_layout()
fig_pem_path = os.path.join(fig_dir, "Grafica_Comparativa_SOEC_vs_PEM_BioSTEAM.png")
plt.savefig(fig_pem_path, dpi=300)
plt.close()
print(f"-> Gráfica generada: {fig_pem_path}")

# Figura 2: Perfiles Axiales del Reactor PFR (1D)
fig, (ax_t, ax_p, ax_c) = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

z = R_PFR.perfil_z
T = R_PFR.perfil_T - 273.15
P = R_PFR.perfil_P / 1e5
f_meoh = (R_PFR.perfil_F[2, :] * 3600.0 * 32.042) / 1000.0 # kg/h
f_co2 = (R_PFR.perfil_F[0, :] * 3600.0 * 44.01) / 1000.0

# Perfil Térmico
ax_t.plot(z, T, color='#C0392B', linewidth=2.5, label='Temperatura del Gas en Lecho (°C)')
ax_t.axhline(240.0, color='#2980B9', linestyle='--', linewidth=1.8, label='Temperatura Agua Ebullición Coraza (240 °C)')
ax_t.scatter([D_pfr['Posicion HotSpot']], [D_pfr['Temperatura Maxima HotSpot']], color='red', s=80, zorder=5)
ax_t.annotate(f"Hot-Spot: {D_pfr['Temperatura Maxima HotSpot']:.2f} °C\nen z = {D_pfr['Posicion HotSpot']:.2f} m",
             xy=(D_pfr['Posicion HotSpot'], D_pfr['Temperatura Maxima HotSpot']),
             xytext=(D_pfr['Posicion HotSpot'] + 0.6, D_pfr['Temperatura Maxima HotSpot'] + 0.8),
             arrowprops=dict(facecolor='black', arrowstyle='->', lw=1.2),
             fontweight='bold', fontsize=10)
ax_t.set_ylabel('Temperatura (°C)', fontweight='bold')
ax_t.set_title('Perfiles Axiales a lo Largo de los Tubos del Reactor PFR (L = 6.0 m, 2,126 tubos)', fontweight='bold', color='#1B365D')
ax_t.legend(loc='upper right')
ax_t.grid(True, linestyle='--', alpha=0.5)

# Perfil de Presión
ax_p.plot(z, P, color='#2C3E50', linewidth=2.5, label='Presión Total del Gas (bar)')
ax_p.set_ylabel('Presión (bar)', fontweight='bold')
ax_p.legend(loc='upper right')
ax_p.grid(True, linestyle='--', alpha=0.5)

# Perfil de Producción y Reactivos
ax_c.plot(z, f_meoh, color='#27AE60', linewidth=2.5, label='Metanol Producido (kg/h)')
ax_c.plot(z, f_co2, color='#7F8C8D', linewidth=2.0, linestyle='-.', label='CO2 Residual (kg/h)')
ax_c.set_xlabel('Posición Axial en Tubo z (m)', fontweight='bold')
ax_c.set_ylabel('Flujo Másico (kg/h)', fontweight='bold')
ax_c.legend(loc='center right')
ax_c.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
fig_pfr_path = os.path.join(fig_dir, "Grafica_Perfiles_Axiales_Reactor_PFR.png")
plt.savefig(fig_pfr_path, dpi=300)
plt.close()
print(f"-> Gráfica generada: {fig_pfr_path}")

print("\nSIMULACIÓN Y COMPARATIVA DE CELDAS PEM Y REACTOR PFR FINALIZADA CON ÉXITO.")